In [89]:
import pandas as pd
import numpy as np

In [91]:
all=pd.read_csv("C:/Users/Administrator/Desktop/채원이/2021_전체 데이터.csv", encoding="cp949")

In [92]:
enrolled= all[all["학적상태"]==1]
expelled= all[all["학적상태"]==0]

In [93]:
enrolled = enrolled.drop(
    columns=["Unnamed: 0", "대학", "학과", "성명"])


In [94]:
expelled = expelled.drop(
    columns=["Unnamed: 0", "대학", "학과", "성명"])

In [57]:
enrolled

,학번,학년,학적상태,입학세부구분,전공2,일반휴학학기수,평점평균,전체취득학점,취업률,강의평가,전임교원수,연구재단등재지(후보포함),SCI급/SCOPUS학술지,등재지,SCI,전임교원_확보율,연구비,혁신수업,계열
96,201810504,2,1,2,0,2,2.94,72,47,4.570,9,28.5000,0.250,3.167,0.028,1,13923.000000,1.015,문과
160,202010238,1,1,1,0,0,2.45,10,47,4.570,9,28.5000,0.250,3.167,0.028,1,13923.000000,1.015,문과
172,202010539,1,1,1,0,0,4.30,16,47,4.570,9,28.5000,0.250,3.167,0.028,1,13923.000000,1.015,문과
292,201710128,2,1,1,0,0,4.23,69,38,4.670,13,14.7917,0.125,1.138,0.010,1,9230.769231,1.012,문과
295,201710139,2,1,2,0,5,3.33,32,38,4.670,13,14.7917,0.125,1.138,0.010,1,9230.769231,1.012,문과
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20023,201713445,3,1,1,0,3,3.68,68,50,4.265,8,2.8096,1.300,0.351,0.163,1,149528.500000,0.027,이과
20109,202013160,1,1,1,0,1,3.80,17,50,4.265,8,2.8096,1.300,0.351,0.163,1,149528.500000,0.027,이과
20127,202013330,1,1,1,0,1,0.00,0,50,4.265,8,2.8096,1.300,0.351,0.163,1,149528.500000,0.027,이과
20170,201510015,2,1,1,0,6,3.49,37,71,4.410,5,8.1250,1.625,1.625,0.325,-1,16933.333333,1.105,미분류


# 범주형

In [95]:
from scipy.stats import chi2_contingency

# 분석할 변수 리스트
vars_to_check = ["학년", "입학세부구분", "전공2", "계열","전임교원_확보율"]  # 원하는 변수명 넣으면 됨
status_col = "학적상태"

In [96]:
results = {}

for var in vars_to_check:
    print(f"==== {var} ====")
    
    #전체 분포
    freq = all[var].value_counts().sort_index()
    percent = all[var].value_counts(normalize=True).sort_index() * 100
    summary = pd.DataFrame({
        "빈도": freq,
        "퍼센트(%)": percent.round(1)
    })
    print("■ 전체 분포")
    print(summary, "\n")
    
    #학적 상태별 교차표 (빈도)
    ct = pd.crosstab(all[var], all[status_col]).sort_index()
    
    #학적 상태별 퍼센트 계산 (열 기준)
    ct_percent = ct.div(ct.sum(axis=0), axis=1) * 100
    
    #마지막 범주 보정
    ct_percent_rounded = ct_percent.round(1)
    for col in ct_percent_rounded.columns:
        diff = 100 - ct_percent_rounded[col].sum()
        last_row = ct_percent_rounded.index[-1]
        ct_percent_rounded.loc[last_row, col] += diff
    
    #빈도 + 퍼센트 문자열 결합
    ct_combined = ct.astype(str) + " (" + ct_percent_rounded.astype(str) + "%)"
    
    print("■ 학적 상태별 교차표 (빈도 + 학적 상태 기준 퍼센트)")
    print(ct_combined, "\n")

    #카이제곱 검정
    chi2, p, dof, expected = chi2_contingency(ct)
    print(f"Chi2 = {chi2:.2f}, p-value = {p:.4f}\n")
    
    #결과 저장
    results[var] = {
        "summary": summary,
        "crosstab": ct,
        "crosstab_with_percent": ct_combined,
        "chi2": chi2,
        "p-value": p,
        "expected": expected
    }


==== 학년 ====
■ 전체 분포
      빈도  퍼센트(%)
학년              
1   4230    20.8
2   5827    28.6
3   4613    22.7
4   5684    27.9 

■ 학적 상태별 교차표 (빈도 + 학적 상태 기준 퍼센트)
학적상태             0                        1
학년                                         
1     3981 (20.0%)              249 (60.4%)
2     5728 (28.7%)               99 (24.0%)
3     4560 (22.9%)               53 (12.9%)
4     5673 (28.4%)  11 (2.699999999999986%) 

Chi2 = 435.16, p-value = 0.0000

==== 입학세부구분 ====
■ 전체 분포
          빈도  퍼센트(%)
입학세부구분              
0       5997    29.5
1       6806    33.4
2       7551    37.1 

■ 학적 상태별 교차표 (빈도 + 학적 상태 기준 퍼센트)
학적상태               0                         1
입학세부구분                                        
0       5931 (29.7%)                66 (16.0%)
1       6585 (33.0%)               221 (53.6%)
2       7426 (37.3%)  125 (30.40000000000001%) 

Chi2 = 82.30, p-value = 0.0000

==== 전공2 ====
■ 전체 분포
        빈도  퍼센트(%)
전공2               
0    18324    90.0
1     2030    10.0 

■ 학적 상태별 

# 연속형

In [97]:
from scipy.stats import ttest_ind
import pandas as pd

# 분석할 연속형 변수 리스트
vars_to_check = ["일반휴학학기수","평점평균","전체취득학점","취업률","강의평가","전임교원수",
                 "연구재단등재지(후보포함)","SCI급/SCOPUS학술지","등재지","SCI","연구비","혁신수업"] 
status_col = "학적상태"

for var in vars_to_check:
    print(f"==== {var} ====")
    
    desc = all[var].describe()
    overall_stats = {
    "최솟값": desc["min"],
    "제1분위수": desc["25%"],
    "중앙값": desc["50%"],
    "평균": desc["mean"],
    "제3분위수": desc["75%"],
    "최댓값": desc["max"],
    "표준편차": desc["std"]
    }
    overall_stats = {k: float(round(v,2)) for k,v in overall_stats.items()}
    print("■ 전체 기초 통계량")
    print(overall_stats)

    
    group_stats = all.groupby(status_col)[var].agg(['mean', 'std', 'count'])
    print("■ 학적 상태별 요약")
    print(group_stats.round(2), "\n")
    
    group1 = all[all[status_col] == 1][var].dropna() #제적
    group2 = all[all[status_col] == 0][var].dropna() #재학

    
    t_stat, p_val = ttest_ind(group1, group2, equal_var=False)
    print(f"■ t-test: t = {t_stat:.3f}, p-value = {p_val:.4f}\n")


==== 일반휴학학기수 ====
■ 전체 기초 통계량
{'최솟값': 0.0, '제1분위수': 0.0, '중앙값': 0.0, '평균': 1.17, '제3분위수': 2.0, '최댓값': 14.0, '표준편차': 1.66}
■ 학적 상태별 요약
      mean   std  count
학적상태                   
0     1.17  1.66  19942
1     0.94  1.56    412 

■ t-test: t = -2.976, p-value = 0.0031

==== 평점평균 ====
■ 전체 기초 통계량
{'최솟값': 0.0, '제1분위수': 3.23, '중앙값': 3.65, '평균': 3.47, '제3분위수': 3.94, '최댓값': 4.5, '표준편차': 0.74}
■ 학적 상태별 요약
      mean   std  count
학적상태                   
0     3.49  0.71  19942
1     2.71  1.51    412 

■ t-test: t = -10.367, p-value = 0.0000

==== 전체취득학점 ====
■ 전체 기초 통계량
{'최솟값': 0.0, '제1분위수': 74.0, '중앙값': 123.0, '평균': 102.53, '제3분위수': 133.0, '최댓값': 201.0, '표준편차': 40.06}
■ 학적 상태별 요약
        mean    std  count
학적상태                      
0     103.99  38.92  19942
1      31.66  28.61    412 

■ t-test: t = -50.366, p-value = 0.0000

==== 취업률 ====
■ 전체 기초 통계량
{'최솟값': 26.0, '제1분위수': 40.0, '중앙값': 47.0, '평균': 49.07, '제3분위수': 56.0, '최댓값': 84.0, '표준편차': 11.68}
■ 학적 상태별 요약
       mean    std  count
학